# 02 — Pré-processamento

Antes de treinar os modelos, verificamos quais ajustes seriam necessários na base de dados. Cada decisão tem a justificativa escrita logo abaixo.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score

RANDOM_STATE = 42
CAMINHO_DADOS = Path('..') / 'data' / 'raw' / 'WineQT.csv'
TARGET = 'qualidade_do_vinho'
pd.set_option('display.max_columns', None)
PROCESSED = Path('..') / 'data' / 'processed'

In [ ]:
df_vinho = pd.read_csv(CAMINHO_DADOS)

## 1. Dados faltantes

In [ ]:
nulos = df_vinho.isnull().sum()
pd.DataFrame({'nulos': nulos, '%': (nulos / len(df_vinho) * 100).round(2)})

**Decisão:** não foram encontrados valores ausentes. Por isso, não foi necessário preencher ou excluir registros por esse motivo.

### Duplicatas

Olhando a base com a coluna Id não aparece nenhuma linha duplicada, porque o Id é diferente em todas. O Id é só a numeração da linha, então tiramos ele e verificamos de novo.

In [ ]:
df_vinho.duplicated().sum()

In [ ]:
df_vinho = df_vinho.drop('Id', axis=1)
df_vinho.duplicated().sum()

In [ ]:
df_vinho = df_vinho.drop_duplicates()
df_vinho.shape

**Decisão:** foram removidas 125 linhas duplicadas, para que o mesmo vinho não caia no treino e no teste ao mesmo tempo.

### Outliers

**Decisão:** os valores extremos foram mantidos, pois não foram identificados motivos suficientes para considerá-los erros de medição. Essa decisão é discutida no notebook 01, seção 3.

## 2. Definição da variável alvo

A coluna quality foi convertida em uma variável binária, separando os vinhos em Alto (nota igual ou superior a 7) e Baixo/Médio (nota inferior a 7).

**Por que o limiar 7:** a distribuição das notas (notebook 01, seção 2.1) mostra que cerca de 83% dos vinhos têm nota 5 ou 6, que representam o vinho comum da base. A partir da nota 7 os vinhos se destacam desse grupo, e nos boxplots por nota é justamente a partir do 7 que alcohol, sulphates, citric acid e volatile acidity mudam de forma mais clara. Um limiar em 6 colocaria mais da metade dos vinhos (cerca de 54%) como Alto, o que não representa uma linha premium. Um limiar em 8 deixaria só 15 vinhos (cerca de 1,5%) na classe Alto, pouco para o modelo aprender.

In [ ]:
def qualidade_do_vinho (qualidade):
  if qualidade < 7:
    return 0  #Qualidade Baixa/Média
  else:
    return 1 #Qualidade Alta

In [ ]:
df_vinho[TARGET] = df_vinho['quality'].apply(qualidade_do_vinho)

In [ ]:
df_vinho.head(5)

In [ ]:
df = df_vinho.drop('quality',axis=1)

## 3. Normalização / padronização

Utilizamos o StandardScaler para colocar as variáveis em escalas comparáveis. Essa etapa é importante porque existem diferenças consideráveis entre os valores das características, como ocorre com total sulfur dioxide, que pode chegar próximo de 300, e density, cujos valores ficam próximos de 1. A Regressão Logística pode ser afetada por essas diferenças de escala, enquanto a Árvore de Decisão e o Random Forest não dependem da padronização da mesma forma.

Para evitar o vazamento de informações, o StandardScaler foi incluído em um Pipeline no notebook 03 (seção 3.3), e não aplicado aqui. Assim, durante a validação cruzada, o ajuste da escala é feito apenas com os dados de treinamento de cada divisão, sem utilizar informações da parte reservada para validação.

## 4. Feature engineering

Optamos por manter as 11 variáveis originais da base, sem criar novas variáveis.

**Justificativa:** as relações mais fortes encontradas na EDA (álcool, acidez volátil, sulfatos e ácido cítrico) já estão representadas diretamente pelas variáveis originais. As combinações que poderiam virar novas variáveis, como álcool com densidade ou SO2 livre com SO2 total, já são variáveis muito ligadas entre si, e o Random Forest consegue combinar variáveis por conta própria. Com cerca de 1.000 vinhos, criar mais variáveis aumentaria o risco de o modelo decorar o treino, e manter as variáveis originais deixa a interpretação dos resultados mais simples para a área de negócio.

## 5. Salvar dataset tratado

In [ ]:
PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED / "dataset_tratado.csv", index=False)